In [2]:
# Importing Initial Libraries


import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import seaborn as sns
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
# importing Libraries


Load Data Into a Dataframe

In [17]:
df = pd.read_csv('/Users/bobyan/Desktop/Kaggle Datasets/hypertension_dataset.csv')
df.columns
df.dtypes

Age                   int64
Salt_Intake         float64
Stress_Score          int64
BP_History           object
Sleep_Duration      float64
BMI                 float64
Medication           object
Family_History       object
Exercise_Level       object
Smoking_Status       object
Has_Hypertension     object
dtype: object

Checking Data

In [5]:
df.sort_values(by=['Age'], ascending=True)

,Age,Salt_Intake,Stress_Score,BP_History,Sleep_Duration,BMI,Medication,Family_History,Exercise_Level,Smoking_Status,Has_Hypertension
568,18,4.7,2,Normal,5.8,30.9,NaN,No,Moderate,Non-Smoker,No
1493,18,10.5,5,Prehypertension,6.7,19.7,NaN,Yes,Low,Non-Smoker,No
339,18,5.9,5,Normal,5.7,31.6,NaN,Yes,Low,Smoker,Yes
1920,18,8.7,0,Hypertension,5.6,29.4,Diuretic,Yes,Moderate,Non-Smoker,Yes
861,18,11.3,3,Prehypertension,7.5,32.4,Other,Yes,High,Non-Smoker,Yes
...,...,...,...,...,...,...,...,...,...,...,...
385,84,8.7,6,Prehypertension,8.5,31.5,NaN,No,High,Smoker,Yes
1191,84,9.9,7,Prehypertension,4.9,20.0,ACE Inhibitor,No,Moderate,Non-Smoker,No
1194,84,9.1,6,Prehypertension,8.1,21.0,Beta Blocker,No,Low,Non-Smoker,No
625,84,8.2,0,Prehypertension,5.9,27.1,ACE Inhibitor,No,Low,Non-Smoker,No


In [10]:
df.columns

Index(['Age', 'Salt_Intake', 'Stress_Score', 'BP_History', 'Sleep_Duration',
       'BMI', 'Medication', 'Family_History', 'Exercise_Level',
       'Smoking_Status', 'Has_Hypertension'],
      dtype='object')

Method 1: Random Forrest and OneHotEncoder

In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

df['Has_Hypertension'].replace("Yes", True, inplace=True)
df['Has_Hypertension'].replace("No", False, inplace=True)
# Drop missing values

# Select features (both numerical + categorical)
#numerical_features = df.select_dtypes(include=['number']).columns.tolist()
#categorical_features = df.select_dtypes(include=['category']).columns.tolist()

categorical_features = df.select_dtypes(include = ['object']).columns.tolist()   # replace with your categorical column names
numerical_features = ['Age', 'Salt_Intake', 'Stress_Score', 'Sleep_Duration', 'BMI']

X = df[numerical_features + categorical_features]
y = df["Has_Hypertension"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Preprocessing: One-hot encode categorical features, keep numerical as is
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Build pipeline with RandomForestRegressor
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)





Evaluation

In [46]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")


Mean Squared Error: 0.04
R^2 Score: 0.82


Method 2: LogisticRegression + OneHotEncoder

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df['Has_Hypertension'].replace("Yes", True, inplace=True)
df['Has_Hypertension'].replace("No", False, inplace=True)

# Example: specify which columns are categorical and which are numeric
categorical_cols = df.select_dtypes(include = ['object']).columns   # replace with your categorical column names
numeric_cols = ['Age', 'Salt_Intake', 'Stress_Score', 'Sleep_Duration', 'BMI']       # replace with your numeric column names

# Preprocessing for categorical data
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Preprocessing for numeric data
numeric_transformer = StandardScaler()

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Build pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model
model.fit(X_train, y_train)
# Predict
y_pred = model.predict(X_test)



Evaluate

In [48]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8823529411764706
Classification Report:
               precision    recall  f1-score   support

       False       0.85      0.87      0.86       101
        True       0.90      0.89      0.90       137

    accuracy                           0.88       238
   macro avg       0.88      0.88      0.88       238
weighted avg       0.88      0.88      0.88       238

